In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Normalization
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import matplotlib as plt
import cv2
import os
import glob

In [3]:
categories = ["fall", "grab", "gun", "hit", "kick", 
              "lying_down", "run", "sit", "sneak", 
              "stand", "struggle", "throw", "walk"]

In [ ]:
video_id = 0

for category in categories:
    video_folder = f"D:/Project/CCTV Motion Sensor Project/Videos/{category}"
    video_files = glob.glob(f"{video_folder}/*.mp4")


    for video_path in video_files:
            output_dir = f"frames/{category}"
            os.makedirs(output_dir, exist_ok=True)

            count = 0
            saved = 0 
            frames = cv2.VideoCapture(video_path)

            while True:
                ret, frame = frames.read()
                if not ret:
                    break
                #cv2.imwrite(f"{output_dir}/frame_{count}.jpg", frame)
                count += 1

                if count % 5 == 0:
                    cv2.imwrite(f"{output_dir}/video_{video_id}_frame_{saved}.jpg", frame)
                    saved += 1

            frames.release()
            video_id += 1

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

dataset_path = "D:/Project/CCTV Motion Sensor Project/frames"

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes :", class_names)
print("Jumlah kelas :", num_classes)

normalization = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

Found 51442 files belonging to 13 classes.
Using 41154 files for training.
Found 51442 files belonging to 13 classes.
Using 10288 files for validation.
Classes : ['fall', 'grab', 'gun', 'hit', 'kick', 'lying_down', 'run', 'sit', 'sneak', 'stand', 'struggle', 'throw', 'walk']
Jumlah kelas : 13


In [ ]:
# data = np.array(data, dtype=np.float32)
# # labels_encoded already computed above

# x_train, x_test, y_train, y_test = train_test_split(
#     data, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
# )

# # convert labels to categorical
# y_train = to_categorical(y_train, num_classes)
# y_test = to_categorical(y_test, num_classes)


MemoryError: Unable to allocate 28.8 GiB for an array with shape (51442, 224, 224, 3) and data type float32

In [ ]:
# normalizer = Normalization()
# normalizer.adapt(x_train)

# x_train = normalizer(x_train)
# x_test = normalizer(x_test)


In [ ]:
# # labels already converted to categorical in previous cell
# print('x_train shape:', x_train.shape)
# print('y_train shape:', y_train.shape)


In [ ]:
# num_classes = len(encoder.classes_)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

NameError: name 'model' is not defined

In [ ]:
loss, acc = model.evaluate(x_test, y_test)
print("Test accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Test accuracy: 1.0


In [ ]:
model.save("model.h5")
np.save("labels.npy", encoder.class_)